# Programación declarativa @ URJC
# Programación funcional
## Curso 23-24, convocatoria ordinaria (30 de octubre de 2023)
## Campus de Vicálvaro


# Definiciones auxiliares

In [5]:
import $ivy.`org.scalatest::scalatest:3.0.8`
import org.scalatest._

import $ivy.$                               

import org.scalatest._

### Algunas definiciones de tipos y funciones auxiliares

In [1]:
sealed abstract class Tree[A]
case class Empty[A]() extends Tree[A]
case class Node[A](left: Tree[A], root: A, right: Tree[A]) extends Tree[A]

object Tree{
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
}

import Tree._

def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B = 
    tree match {
        case Empty() => empty
        case Node(left, root, right) => node(foldTree(left)(empty)(node), root, foldTree(right)(empty)(node))
    }

defined class Tree
defined class Empty
defined class Node
defined object Tree
import Tree._


defined function foldTree

In [2]:
object Signatures{
    abstract class List[A]{
        
        // Common HOFs
        def foldRight[B](nil: B)(cons: (A, B) => B): B
        def foldLeft[B](initial: B)(update: (B, A) => B): B
        def map[B](f: A => B): List[B]
        def flatMap[B](f: A => List[B]): List[B]
        def filter(f: A => Boolean): List[A]
        def forall(pred: A => Boolean): Boolean
        def exists(pred: A => Boolean): Boolean
 
        // Returns the number of elements of this list
        def length: Int
        def reverse: List[A]
    }
}

defined object Signatures

### Definiciones auxiliares sobre la correspondencia Curry-Howard y Tipos Algebraicos de Datos

In [3]:
trait Isomorphic[A, B]{
    
    def from(a: A): B
    
    def to(b: B): A
}

defined trait Isomorphic

In [4]:
type Not[P] = P => Nothing

defined type Not

# Ejercicio 1
__(2 puntos)__

__a) (1 punto)__ Utiliza la correspondencia de Curry-Howard para demostrar la validez del siguiente argumento lógico: 

$ \{ r \rightarrow s, p \vee q, \neg(\neg p \rightarrow s) \} \vdash q \wedge \neg r$

In [61]:
def proof[P, Q, R, S](p1: R => S, p2: Either[P, Q], p3: Not[Not[P] => S]): (Q, Not[R]) = 
    p2 match {
        case Left(p) => 
            (p3((np: Not[P]) => np(p)) : Q, (r: R) => p3((np: Not[P]) => p1(r) : S))
        case Right(q) => 
            (q : Q, (r: R) => p3((np: Not[P]) => p1(r) : S))
    }

defined function proof

__b) (1 punto)__ Utiliza la correspondencia Curry-Howard para demostrar la siguiente tautología de la lógica clásica proposicional: $\vdash_c (\neg p \rightarrow b) \rightarrow \neg q \rightarrow a$. Para ello, utiliza como premisa adicional la ley de la doble negación aplicada a la proposición $p$, es decir, demuestra la validez intuicionista del siguiente argumento lógico:

$ \{ \neg\neg p \rightarrow p \} \vdash (\neg p \rightarrow q) \rightarrow \neg q \rightarrow p $

In [64]:
def proof[P, Q](dn: Not[Not[P]] => P): (Not[P] => Q) => Not[Q] => P = 
    (f: Not[P] => Q) => (nb: Not[Q]) => 
        dn( (na: Not[P]) => nb(f(na)))

defined function proof

# Ejercicio 2
__(1 punto)__

Dada la definición del tipo de datos `Either4` mostrada a continuación: 

In [51]:
sealed abstract class Either4[A, B, C, D]
case class Item1[A, B, C, D](a: A) extends Either4[A, B, C, D]
case class Item2[A, B, C, D](b: B) extends Either4[A, B, C, D]
case class Item3[A, B, C, D](c: C) extends Either4[A, B, C, D]
case class Item4[A, B, C, D](d: D) extends Either4[A, B, C, D]

defined class Either4
defined class Item1
defined class Item2
defined class Item3
defined class Item4

demuestra que, para todo tipo `A`, `B`, `C` y `D`, los tipos de datos `(Either[A, B], Either[C, D])` y `Either4[(A, C), (A, D), (B, C), (B, D)]` son isomorfos:

In [52]:
class Iso[A, B, C, D] extends Isomorphic[(Either[A, B], Either[C, D]), Either4[(A, C), (A, D), (B, C), (B, D)]]{

    def from(t: (Either[A, B], Either[C, D])): Either4[(A, C), (A, D), (B, C), (B, D)] = 
        t match {
            case (Left(a), Left(c)) => Item1((a,c))
            case (Left(a), Right(d)) => Item2((a,d))
            case (Right(b), Left(c)) => Item3((b,c))
            case (Right(b), Right(d)) => Item4((b,d))
        }
            
    def to(t: Either4[(A, C), (A, D), (B, C), (B, D)]): (Either[A, B], Either[C, D]) = 
        t match {
            case Item1((a,c)) => (Left(a), Left(c))
            case Item2((a,d)) => (Left(a), Right(d))
            case Item3((b,c)) => (Right(b), Left(c))
            case Item4((b,d)) => (Right(b), Right(d))
        }
}

defined class Iso

# Ejercicio 3
__(4 puntos)__




Dada una lista de elementos, la función `tails` devuelve una lista formada por todas sus sublistas, incluida a ella misma, en el orden que se indica en los siguientes ejemplos: 

In [41]:
class TestSorted(isSorted: List[Int] => Boolean) extends FlatSpec with Matchers{
    
    "sorted" should "work" in {
        
        isSorted(Nil) shouldBe true
        isSorted(List(1)) shouldBe true
        isSorted(List(0,1)) shouldBe true
        isSorted(List(0,1,2,3,4)) shouldBe true
        isSorted(List(1,0)) shouldBe false
        isSorted(List(1,2,0)) shouldBe false
        isSorted(List(3,1,2)) shouldBe false
    }
}

defined class TestSorted

__a) (1 punto)__ Implementa la función `tails` recursivamente, mediante un patrón de divide y vencerás.

In [44]:
def isSorted[A](l: List[A])(lessEq: (A, A) => Boolean): Boolean = {
    
    def sortedAux(out: Boolean, last: Option[A], aux: List[A]): Boolean = 
        aux match {
            case Nil => out
            case h1 :: t => 
                val sortedLast = last match {
                    case None => true
                    case Some(h2) => lessEq(h2, h1)
                }
                sortedAux(out && sortedLast, Some(h1), t)
        }
    
    sortedAux(true, None, l)
}

defined function isSorted

In [45]:
run(new TestSorted(isSorted(_)(_ <= _)))

cmd40$Helper$TestSorted:
sorted
- should work


__b) (1 punto)__ Implementa la función `tails` mediante la función `foldRight`. 

In [46]:
def isSorted[A](l: List[A])(lessEq: (A, A) => Boolean): Boolean =
    l.foldLeft((true, None: Option[A])){
        case ((_, None), e) => (true, Some(e))
        case ((isSorted, Some(last)), e) => (isSorted && lessEq(last, e), Some(e))
    }._1

defined function isSorted

In [47]:
run(new TestSorted(isSorted(_)(_ <= _)))

cmd40$Helper$TestSorted:
sorted
- should work


# Ejercicio 4
__(4 puntos)__

Dadas las funciones `zip` y `range` especificadas a través de los siguientes ejemplos:

In [23]:
class TestHeads(heads: Tree[List[Int]] => Option[Tree[Int]]) extends FlatSpec with Matchers{
    
    // 
    // 
    
    "heads" should "work" in {
        heads(node(leaf(List(1,2,3)), List(5), leaf(List(10,0)))) shouldBe 
            Some(node(leaf(1), 5, leaf(10)))
        
        heads(node(leaf(Nil), List(5,3), leaf(List(10)))) shouldBe 
            None
        
        heads(node(leaf(List(1,2)), List(), leaf(List(10)))) shouldBe 
            None
        
        heads(node(leaf(List(1,2)), List(5,3), leaf(List()))) shouldBe 
            None
        
        heads(node(node(leaf(List(1,2)), List(6), leaf(List(3,4,2))), List(5), leaf(List(2,3,2)))) shouldBe 
            Some(node(node(leaf(1), 6, leaf(3)), 5, leaf(2)))        
    }
}

defined class TestHeads

In [24]:
class TestDivide(divide: Int => Tree[Int] => Option[Tree[Int]]) extends FlatSpec with Matchers{
    
    // Dado un valor máximo `max` y un valor mínimo `min`, la expresión `range(max)(min)` 
    // consiste en una lista que representa el rango de valores min..max.
    
    "divide" should "work" in {
        divide(10)(node(leaf(1), 5, leaf(10))) shouldBe 
            Some(node(leaf(10), 2, leaf(1)))
        
        divide(10)(node(leaf(0), 5, leaf(10))) shouldBe 
            None
        
        divide(10)(node(leaf(1), 5, leaf(0))) shouldBe 
            None
        
        divide(10)(node(leaf(1), 0, leaf(10))) shouldBe 
            None
        
        divide(30)(node(node(leaf(1), 6, leaf(3)), 5, leaf(2))) shouldBe 
            Some(node(node(leaf(30), 5, leaf(10)), 6, leaf(15)))        
    }
    
}

defined class TestDivide

Considérense las siguientes implementaciones basadas en un mismo patrón de diseño:

In [25]:
def heads[A](t: Tree[List[A]]): Option[Tree[A]] = 
    t match {
        case Empty() => 
            Some(Empty())
        case Node(left, b, right) => 
            (heads(left), b match { case Nil => None; case h :: _ => Some(h) }, heads(right)) match {
                case (Some(left), Some(r), Some(right)) => 
                    Some(Node(left, r, right))
                case _ => 
                    None
            }
    }

defined function heads

In [26]:
run(new TestHeads(heads))

cmd22$Helper$TestHeads:
heads
- should work


In [27]:
def divide(a: Int)(t: Tree[Int]): Option[Tree[Int]] = 
    t match {
        case Empty() => 
            Some(Empty())
        case Node(left, b, right) => 
            (divide(a)(left), if (b == 0) None else Some(a/b), divide(a)(right)) match {
                case (Some(left), Some(r), Some(right)) => 
                    Some(Node(left, r, right))
                case _ => 
                    None
            }
    }

defined function divide

In [28]:
run(new TestDivide(divide))

cmd23$Helper$TestDivide:
divide
- should work


__a) (2 puntos)__ Implementa la función de orden superior `unfold`, cuya signatura se muestra a continuación. Esta función encapsula el patrón de diseño utilizado en las implementaciones anteriores de las funciones `zip` y `range`. 

In [29]:
def traverse[A, B](t: Tree[A])(f: A => Option[B]): Option[Tree[B]] = 
    t match {
        case Empty() => 
            Some(Empty())
        case Node(left, a, right) => 
            (traverse(left)(f), f(a), traverse(right)(f)) match {
                case (Some(left), Some(r), Some(right)) => 
                    Some(Node(left, r, right))
                case _ => 
                    None
            }
    }

defined function traverse

__b) (2 puntos)__ Implementa las funciones `zip` y `range` mediante la función `unfold`.

In [34]:
def heads[A](t: Tree[List[A]]): Option[Tree[A]] = 
    traverse(t){ 
        case Nil => None
        case h :: _ => Some(h) 
    }

defined function heads

In [35]:
run(new TestHeads(heads))

cmd22$Helper$TestHeads:
heads
- should work


In [36]:
def divide(a: Int)(t: Tree[Int]): Option[Tree[Int]] = 
    traverse(t)(b => if (b == 0) None else Some(a/b))

defined function divide

In [37]:
run(new TestDivide(divide))

cmd23$Helper$TestDivide:
divide
- should work


__c) (1 punto)__

In [38]:
def sequence[A](t: Tree[Option[A]]): Option[Tree[A]] = 
    t match {
        case Empty() =>
            Some(Empty())
        case Node(left, a, right) => 
            (sequence(left), a, sequence(right)) match {
                case (Some(left), Some(a), Some(right)) => 
                    Some(Node(left, a, right))
                case _ => 
                    None
            }
    }

defined function sequence

In [39]:
def sequence[A](t: Tree[Option[A]]): Option[Tree[A]] = 
    traverse(t)(identity)

defined function sequence